In [ ]:
import datetime
import fabric.functions as fn
import logging

udf = fn.UserDataFunctions()

In [ ]:
## 1. Funciones para la tabla D_Producto

In [ ]:
# 1.1 Eliminar registro indicado de tabla Producto
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Eliminar_producto(sqlDB: fn.FabricSqlConnection,
                      idunico: int) -> str:
    
    connection = sqlDB.connect()
    cursor = connection.cursor()

    if idunico:

        # Comprobar si el producto está en la tabla de presupuesto
        check_query = """
            SELECT TOP 1 1
            FROM [dbo].[f_mov_budget]
            WHERE [ProductoID] = ?
        """
        cursor.execute(check_query, (idunico,))
        exists = cursor.fetchone()

        if exists:
            raise fn.UserThrownError(
                "No se puede eliminar el producto porque tiene movimientos en presupuesto.",
                {"IDUNICO": idunico}
            )

        # Si no existe (borrar el registro)
        delete_query = """
            DELETE FROM [dbo].[d_productos]
            WHERE [ID] = ?
        """
        cursor.execute(delete_query, (idunico,))

        if cursor.rowcount == 0:
            raise fn.UserThrownError(
                "No se encontró el ID para eliminar.",
                {"IDUNICO": idunico}
            )

    connection.commit()
    cursor.close()
    connection.close()

    return f"Eliminado correctamente el registro con Id {idunico}."

In [ ]:
# 1.2. Actualizar campo "ProductoNombre" del registro
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def actualizar_nombre_producto(sqlDB: fn.FabricSqlConnection,
                         idunico: int,
                         nombreproducto: str) -> str:

    if not nombreproducto :
        raise fn.UserThrownError("Todos los campos (id, nombre de producto) son obligatorios.", {
            "nombre producto": nombreproducto
        })

    connection = sqlDB.connect()
    cursor = connection.cursor()

    update_query = """
        UPDATE [dbo].[d_productos]
        SET [ProductoNombre] = ?
        WHERE [ID] = ?
    """

    cursor.execute(update_query, (nombreproducto, idunico))

    if cursor.rowcount == 0:
        raise fn.UserThrownError("No se encontró ningún registro con ese Id.", {"ID": idunico})

    connection.commit()
    cursor.close()
    connection.close()

    return f"Registro actualizado correctamente para Id {idunico}."

In [ ]:
# 1.3. Insertar nuevo registro (Tabla Producto)
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Insertar_registro_producto(sqlDB: fn.FabricSqlConnection,
                       productoid: int,
                       productonombre: str,
                       subcategoriaid: int,
                       preciounitario: int,
                       unidadmedida: str,
                       productoidnombre: str) -> str:               

    if not productoid or not productonombre or not subcategoriaid or not preciounitario or not unidadmedida or not productoidnombre:
        raise fn.UserThrownError("Todos los campos a rellenar son obligatorios.", {
            "ProductoID": productoid,
            "ProductoNombre": productonombre,
            "SubcategoriaID": subcategoriaid,
            "PrecioUnitario": preciounitario,
            "UnidadMedida": unidadmedida,
            "ProductoIDNombre": productoidnombre,

        })
    
    insert_query = """
        INSERT INTO [dbo].[d_productos] (
            [ProductoID],
            [ProductoNombre],
            [SubcategoriaID],
            [PrecioUnitario],
            [UnidadMedida],
            [ProductoIDNombre]
        ) VALUES (?, ?, ?, ?, ?, ?)
    """

   # Establecer conexión y ejecutar la consulta
    connection = sqlDB.connect()
    cursor = connection.cursor()
    try:
        cursor.execute(insert_query, (productoid, productonombre, subcategoriaid, preciounitario, unidadmedida, productoidnombre))
        connection.commit()
    finally:
        connection.close()

    return "Se insertó el nuevo registro correctamente."

In [ ]:
# 1.4. Copiar un registro existente
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Copiar_registro_producto(sqlDB: fn.FabricSqlConnection, idunicoorigen: int) -> str:
    if not idunicoorigen:
        raise fn.UserThrownError(
            "El campo ID de origen es obligatorio.",
            {"ID": idunicoorigen}
        )

    connection = sqlDB.connect()
    cursor = connection.cursor()

    try:
        copy_query = """
            INSERT INTO [dbo].[d_productos] (
                [ProductoID],
                [ProductoNombre],
                [SubcategoriaID],
                [PrecioUnitario],
                [UnidadMedida],
                [ProductoIDNombre]
            )
            OUTPUT INSERTED.[ID]
            SELECT
                [ProductoID],
                [ProductoNombre],
                [SubcategoriaID],
                [PrecioUnitario],
                [UnidadMedida],
                [ProductoIDNombre]
            FROM [dbo].[d_productos]
            WHERE [ID] = ?
        """

        cursor.execute(copy_query, (idunicoorigen,))
        row = cursor.fetchone()

        if not row:
            raise fn.UserThrownError(
                "No se encontró el registro origen para copiar.",
                {"ID": idunicoorigen}
            )

        nuevo_id = row[0]
        connection.commit()

    finally:
        cursor.close()
        connection.close()

    return f"Registro copiado correctamente. ID origen: {idunicoorigen}. Nuevo ID: {nuevo_id}."

In [ ]:
## ----------------------------------------------------------------------------------

In [ ]:
## 2. Funciones para la tabla D_Cliente

In [ ]:
# 2.1 Eliminar registro indicado de tabla Cliente
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Eliminar_cliente(sqlDB: fn.FabricSqlConnection,
                                           idunico: int ) -> str:
    
    connection = sqlDB.connect()
    cursor = connection.cursor()

    
    if idunico:
        delete_query = """
            DELETE FROM [dbo].[d_clientes]
            WHERE [ID] = ?
        """
        cursor.execute(delete_query, (idunico,))
        if cursor.rowcount == 0:
            raise fn.UserThrownError("No se encontró el ID para eliminar.", {"IDUNICO": idunico})

    
    connection.commit()
    cursor.close()
    connection.close()

    return f"Eliminado correctamente el registro con Id {idunico}."

In [ ]:
# 2.2. Actualizar campo "TipoCliente" del registro
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def actualizar_tipo_cliente(sqlDB: fn.FabricSqlConnection,
                         idunico: int,
                         tipocliente: str) -> str:

    if not tipocliente :
        raise fn.UserThrownError("Todos los campos (id, tipo de cliente) son obligatorios.", {
            "nombre producto": tipocliente
        })

    connection = sqlDB.connect()
    cursor = connection.cursor()

    update_query = """
        UPDATE [dbo].[d_clientes]
        SET [TipoCliente] = ?
        WHERE [ID] = ?
    """

    cursor.execute(update_query, (tipocliente, idunico))

    if cursor.rowcount == 0:
        raise fn.UserThrownError("No se encontró ningún registro con ese Id.", {"ID": idunico})

    connection.commit()
    cursor.close()
    connection.close()

    return f"Registro actualizado correctamente para Id {idunico}."

In [ ]:
# 2.3. Insertar nuevo registro (Tabla Producto)
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Insertar_registro_cliente(sqlDB: fn.FabricSqlConnection,
                       clienteid: int,
                       clientenombre: str,
                       tipocliente: str,
                       provinciaid: int) -> int:               

    if not clienteid or not clientenombre or not tipocliente or not provinciaid:
        raise fn.UserThrownError("Todos los campos a rellenar son obligatorios.", {
            "ClienteID": clienteid,
            "ClienteNombre": clientenombre,
            "TipoCliente": tipocliente,
            "ProvinciaID": provinciaid
        })
    
    insert_query = """
        INSERT INTO [dbo].[d_clientes] (
            [ClienteID],
            [ClienteNombre],
            [TipoCliente],
            [ProvinciaID]
        ) VALUES (?, ?, ?, ?)
    """

   # Establecer conexión y ejecutar la consulta
    connection = sqlDB.connect()
    cursor = connection.cursor()
    try:
        cursor.execute(insert_query, (clienteid, clientenombre, tipocliente, provinciaid))
        connection.commit()
    finally:
        connection.close()

    return "Se insertó el nuevo registro correctamente."

In [ ]:
# 2.4. Duplicar un registro existente
@udf.connection(argName="sqlDB", alias="dwsatlanticwood")
@udf.function()
def Copiar_registro_cliente(sqlDB: fn.FabricSqlConnection, idunicoorigen: int) -> str:
    if not idunicoorigen:
        raise fn.UserThrownError(
            "El campo ID de origen es obligatorio.",
            {"ID": idunicoorigen}
        )

    connection = sqlDB.connect()
    cursor = connection.cursor()

    try:
        copy_query = """
            INSERT INTO [dbo].[d_clientes] (
                [ClienteID],
                [ClienteNombre],
                [TipoCliente],
                [ProvinciaID]
            )
            OUTPUT INSERTED.[ID]
            SELECT
                [ClienteID],
                [ClienteNombre],
                [TipoCliente],
                [ProvinciaID]
            FROM [dbo].[d_clientes]
            WHERE [ID] = ?
        """

        cursor.execute(copy_query, (idunicoorigen,))
        row = cursor.fetchone()

        if not row:
            raise fn.UserThrownError(
                "No se encontró el registro origen para copiar.",
                {"ID": idunicoorigen}
            )

        nuevo_id = row[0]
        connection.commit()

    finally:
        cursor.close()
        connection.close()

    return f"Registro copiado correctamente. ID origen: {idunicoorigen}. Nuevo ID: {nuevo_id}."